In [2]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'r') as f:
    src = f.read()

print(f"File letto: {len(src)} caratteri")

# Trova tutte le carte con regex robusta
card_re = re.compile(
    r"(effects:\{[^}]*\})",
    re.DOTALL
)

# Prima analizziamo la struttura per capire il formato
# Cerca card_id per capire quante carte ci sono
card_ids = re.findall(r"card_id:'([^']+)'", src)
fazioni = re.findall(r"faction:'([^']+)'", src)
tipi = re.findall(r"card_type:'([^']+)'", src)

print(f"Carte trovate: {len(card_ids)}")
print(f"Fazioni: {set(fazioni)}")
print(f"Tipi: {set(tipi)}")

# Conta per fazione+tipo
from collections import Counter
combo = Counter(zip(fazioni, tipi))
for k,v in sorted(combo.items()):
    print(f"  {k[0]} / {k[1]}: {v}")


File letto: 87354 caratteri
Carte trovate: 344
Fazioni: {'Coalizione', 'Cina', 'Neutrale', 'Europa', 'Iran', 'Russia'}
Tipi: {'Evento', 'Media', 'Militare', 'Politico', 'Segreto', 'Economico', 'Diplomatico'}
  Cina / Diplomatico: 18
  Cina / Economico: 25
  Cina / Media: 2
  Cina / Militare: 8
  Cina / Politico: 1
  Cina / Segreto: 4
  Coalizione / Diplomatico: 25
  Coalizione / Economico: 13
  Coalizione / Media: 3
  Coalizione / Militare: 18
  Coalizione / Politico: 2
  Coalizione / Segreto: 9
  Europa / Diplomatico: 26
  Europa / Economico: 25
  Europa / Media: 1
  Europa / Militare: 4
  Europa / Politico: 2
  Iran / Diplomatico: 16
  Iran / Economico: 13
  Iran / Media: 4
  Iran / Militare: 25
  Iran / Politico: 6
  Iran / Segreto: 6
  Neutrale / Diplomatico: 12
  Neutrale / Economico: 4
  Neutrale / Evento: 7
  Neutrale / Media: 1
  Neutrale / Militare: 2
  Neutrale / Politico: 3
  Neutrale / Segreto: 1
  Russia / Diplomatico: 17
  Russia / Economico: 15
  Russia / Media: 5
  Russ

In [5]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'r') as f:
    src = f.read()

# Mostra le prime 3 carte per capire la struttura
sample = src[:3000]
print(sample)


// =============================================
// LINEA ROSSA — Mazzi completi per fazione
// Iran: 24 carte base (C025-C048)
// Coalizione: 24 carte base (C001-C024)
// Russia: 18 carte (C073-C095 dispari Russia)
// Cina: 18 carte (dalla sezione Russia-Cina condivisa)
// Europa: 18 carte (C097-C120)
// Neutrale: 18 carte evento generali
// =============================================

import type { GameCard } from '@/types/game';

// Helper: clamp valore in range
const clamp = (val: number, min: number, max: number) => Math.max(min, Math.min(max, val));

// -----------------------------------------------
// MAZZO IRAN (24 carte base)
// -----------------------------------------------
export const MAZZO_IRAN: GameCard[] = [
  { card_id:'C025', card_name:'Arricchimento Uranio 60%', faction:'Iran', card_type:'Militare', op_points:4, deck_type:'base', description:'Accelerazione del programma nucleare',
    effects:{ nucleare:(v)=>v<=7?2:v<=12?1:3, sanzioni:(v)=>v>=5?-1:0, defcon:(v)=>v

In [8]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'r') as f:
    src = f.read()

# Regex per trovare ogni carta intera (dalla { alla chiusura effects})
# Il pattern cerca: card_id, faction, card_type, poi il blocco effects
CARD_RE = re.compile(
    r"""(\{[ \t]*card_id:'(?P<cid>[^']+)',[ \t]*card_name:'[^']*',[ \t]*faction:'(?P<fac>[^']+)',[ \t]*card_type:'(?P<ctp>[^']+)',[ \t]*op_points:\d+,[ \t]*deck_type:'[^']*',[ \t]*description:'[^']*',?[ \t]*(?:unlocks_special:[ \t]*true,?[ \t]*)?)(?P<eff>effects:\{[^}]*\})""",
    re.DOTALL
)

def get_new_effects(faction, card_type, eff_str):
    """Calcola nuovi effetti da aggiungere secondo le regole narrative."""
    new_eff = {}

    if faction == 'Iran':
        if card_type == 'Militare':
            new_eff['forze_militari_iran'] = 1
            new_eff['stabilita_coalizione'] = -1
            # Se ha nucleare positivo → tecnologia_nucleare_iran
            nm = re.search(r'nucleare:\(v\)=>([^,}]+)', eff_str)
            if nm:
                try:
                    v_test = eval(f"(lambda v:{nm.group(1).strip()})(5)")
                    if v_test > 0:
                        new_eff['tecnologia_nucleare_iran'] = 1
                except:
                    pass
        elif card_type == 'Diplomatico':
            new_eff['influenza_diplomatica_coalizione'] = -1
            new_eff['supporto_pubblico_coalizione'] = -1
        elif card_type == 'Economico':
            new_eff['risorse_coalizione'] = -1
        elif card_type in ('Segreto', 'Media'):
            new_eff['stabilita_iran'] = 1

    elif faction == 'Coalizione':
        if card_type == 'Militare':
            new_eff['forze_militari_iran'] = -1
            new_eff['tecnologia_nucleare_iran'] = -1
        elif card_type == 'Diplomatico':
            new_eff['influenza_diplomatica_coalizione'] = 1
            new_eff['supporto_pubblico_coalizione'] = 1
        elif card_type == 'Economico':
            new_eff['risorse_iran'] = -1
            new_eff['stabilita_iran'] = -1
        elif card_type == 'Segreto':
            new_eff['cyber_warfare_cina'] = -1
            new_eff['tecnologia_nucleare_iran'] = -1

    elif faction == 'Russia':
        if card_type == 'Militare':
            new_eff['influenza_militare_russia'] = 1
            new_eff['stabilita_coalizione'] = -1
        elif card_type == 'Diplomatico':
            new_eff['veto_onu_russia'] = 1
            new_eff['influenza_militare_russia'] = 1
        elif card_type == 'Economico':
            new_eff['stabilita_economica_russia'] = 1
            new_eff['risorse_iran'] = 1

    elif faction == 'Cina':
        if card_type in ('Militare', 'Segreto'):
            new_eff['cyber_warfare_cina'] = 1
            new_eff['influenza_commerciale_cina'] = 1
        elif card_type == 'Diplomatico':
            new_eff['influenza_commerciale_cina'] = 1
            new_eff['coesione_ue_europa'] = -1
        elif card_type == 'Economico':
            new_eff['risorse_cina'] = 1
            new_eff['stabilita_rotte_cina'] = 1
            new_eff['influenza_commerciale_cina'] = 1

    elif faction == 'Europa':
        if card_type == 'Diplomatico':
            new_eff['influenza_diplomatica_europa'] = 1
            new_eff['coesione_ue_europa'] = 1
            new_eff['aiuti_umanitari_europa'] = 1
        elif card_type == 'Economico':
            new_eff['risorse_europa'] = 1
            new_eff['coesione_ue_europa'] = 1
        elif card_type == 'Militare':
            new_eff['stabilita_coalizione'] = 1
            new_eff['influenza_diplomatica_europa'] = 1

    elif faction == 'Neutrale':
        nm = re.search(r'nucleare:\(v\)=>([^,}]+)', eff_str)
        if nm:
            try:
                v_test = eval(f"(lambda v:{nm.group(1).strip()})(5)")
                new_eff['tecnologia_nucleare_iran'] = 1 if v_test > 0 else -1
            except:
                pass
        dm = re.search(r'defcon:\(v\)=>([^,}]+)', eff_str)
        if dm:
            try:
                v_test = eval(f"(lambda v:{dm.group(1).strip()})(7)")
                if v_test != 0:
                    new_eff['stabilita_coalizione'] = 1 if v_test > 0 else -1
            except:
                pass

    # Filtra quelli già presenti
    return {k: v for k, v in new_eff.items() if f'{k}:' not in eff_str}


# Applica modifiche
new_src = src
offset = 0
count_total = 0
count_by_faction = defaultdict(int)

for m in CARD_RE.finditer(src):
    faction = m.group('fac')
    card_type = m.group('ctp')
    card_id = m.group('cid')
    eff_str = m.group('eff')

    new_eff = get_new_effects(faction, card_type, eff_str)

    if new_eff:
        # Costruisci il nuovo effects string
        new_eff_str = eff_str[:-1]  # rimuovi '}'
        for k, v in new_eff.items():
            new_eff_str += f', {k}:(v)=>{v}'
        new_eff_str += '}'

        pos = m.start('eff') + offset
        new_src = new_src[:pos] + new_eff_str + new_src[pos + len(eff_str):]
        offset += len(new_eff_str) - len(eff_str)
        count_total += 1
        count_by_faction[faction] += 1

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'w') as f:
    f.write(new_src)

print(f"=== CARTE AGGIORNATE: {count_total} ===")
for fac, cnt in sorted(count_by_faction.items()):
    print(f"  {fac}: {cnt}")


=== CARTE AGGIORNATE: 0 ===
